# Multi-Host Distributed PyTorch Training with TorchTPU on GKE

This notebook demonstrates how to launch a **multi-host distributed PyTorch training job** across a multi-host Cloud TPU slice (e.g. 2 hosts × 4 chips = 8 TPU v5e chips) using **Kubeflow Trainer (`TrainJob`)**, **TorchTPU (`torch_tpu`)**, and **`torchrun`** on Google Kubernetes Engine (GKE).

---

## How Multi-Host TorchTPU Works on GKE

In a multi-host TPU slice (such as a `2x4` TPU v5e slice), multiple Kubernetes Pods are scheduled across distinct TPU host VMs connected by Google's high-bandwidth Inter-Chip Interconnect (ICI).

```mermaid
flowchart TD
    subgraph Notebook["Interactive Workspace (CPU or Single TPU)"]
      NB["JupyterLab / VS Code<br/>Submit TrainJob via Kubeflow SDK"]
    end

    subgraph TrainerControl["Kubeflow Trainer & GKE"]
      TJ["TrainJob (JobSet)<br/>Injects PET_NNODES, PET_NODE_RANK, PET_MASTER_ADDR/PORT"]
      GCW["GKE Common Webhooks (GCW)<br/>Injects TPU_WORKER_HOSTNAMES, TPU_HOST_BOUNDS, TPU_CHIPS_PER_HOST_BOUNDS"]
    end

    subgraph Node0["TPU Host 0 (trainer-node-0-0)"]
      TRUN0["torchrun --nproc_per_node=4"]
      INIT0["_maybe_init_distributed_on_gke()<br/>Configures SliceBuilder & 3D Topology per Worker"]
      R0["Rank 0 (TPU Core 0)"]
      R1["Rank 1 (TPU Core 1)"]
      R2["Rank 2 (TPU Core 2)"]
      R3["Rank 3 (TPU Core 3)"]
    end

    subgraph Node1["TPU Host 1 (trainer-node-0-1)"]
      TRUN1["torchrun --nproc_per_node=4"]
      INIT1["_maybe_init_distributed_on_gke()<br/>Configures SliceBuilder & 3D Topology per Worker"]
      R4["Rank 4 (TPU Core 4)"]
      R5["Rank 5 (TPU Core 5)"]
      R6["Rank 6 (TPU Core 6)"]
      R7["Rank 7 (TPU Core 7)"]
    end

    NB -->|"client.train(...)"| TJ
    TJ --> GCW
    GCW --> Node0
    GCW --> Node1
    TRUN0 --> INIT0 --> R0 & R1 & R2 & R3
    TRUN1 --> INIT1 --> R4 & R5 & R6 & R7
    R0 <== "ICI Network (tpu_dist AllReduce)" ==> R4
    R1 <== "ICI Network (tpu_dist AllReduce)" ==> R5
    R2 <== "ICI Network (tpu_dist AllReduce)" ==> R6
    R3 <== "ICI Network (tpu_dist AllReduce)" ==> R7
```

### The Environment Bootstrap Contract

To coordinate across multi-host TPU slices on GKE, each `torchrun` worker calls `_maybe_init_distributed_on_gke()` before initializing `torch.distributed`:
1. **`TPU_WORKER_HOSTNAMES`, `TPU_HOST_BOUNDS`, & `TPU_CHIPS_PER_HOST_BOUNDS`**: Automatically injected by **GKE Common Webhooks (GCW)** for Indexed Jobs requesting TPUs (e.g. `TPU_HOST_BOUNDS=1,2,1` and `TPU_CHIPS_PER_HOST_BOUNDS=2,2,1` for a `2x4` slice).
2. **`_maybe_init_distributed_on_gke()`**: Combines the GKE host/chip bounds with `torchrun`'s per-worker `WORLD_SIZE`, `LOCAL_RANK`, and `RANK` to configure:
   - **`TPU_HOST_BOUNDS` & `TORCH_TPU_TOPOLOGY`**: Multiplies host bounds by chip-per-host bounds to derive the full 3D slice topology (`2,4,1`, or 4D with `,2` on dual-core `tpu7x`) and sets per-process `TPU_CHIPS_PER_HOST_BOUNDS=1,1,1`.
   - **`TORCH_TPU_SLICEBUILDER_ADDRESSES` & `TPU_PROCESS_ADDRESSES`**: Builds `<hostname>:<port>` mesh endpoints across all hosts and per-host worker ports (`10000 + local_rank`).
   - **`TPU_VISIBLE_CHIPS`, `CLOUD_TPU_TASK_ID`, & `TPU_PROCESS_PORT`**: Binds each worker process to its dedicated local TPU chip (`local_rank`), global task ID (`rank`), and SliceBuilder port.
3. **PyTorch Rendezvous (`PET_NNODES`, `PET_NODE_RANK`, `PET_MASTER_ADDR`, `PET_MASTER_PORT`)**: Automatically injected by Kubeflow Trainer's `torch-distributed` runtime and read natively by `torchrun`.

## 1. Setup & Environment Verification

Discover the current Kubernetes namespace and verify access to the cluster.

In [ ]:
import os
import pathlib
import sys
import time
from kubernetes import client as k8s_client, config as k8s_config

# Load in-cluster or local kubeconfig
try:
    k8s_config.load_incluster_config()
except Exception:
    k8s_config.load_kube_config()

def get_current_namespace() -> str:
    if os.environ.get("TENANT_NAMESPACE"):
        return os.environ["TENANT_NAMESPACE"]
    ns_path = "/var/run/secrets/kubernetes.io/serviceaccount/namespace"
    if os.path.exists(ns_path):
        with open(ns_path) as f:
            return f.read().strip()
    return "default"

namespace = get_current_namespace()
print(f"Target Kubernetes namespace: {namespace}")

## 2. Define Modular Training Library and Entrypoint (`src/`)

In realistic machine learning projects, code is organized across multiple library modules rather than combined into a single monolithic script:
- **`src/models.py`**: Neural network architecture definitions (`MLPClassifier` using PyTorch `nn.Module`).
- **`src/dataset.py`**: Synthetic data generation, preprocessing, and distributed `DataLoader` / `DistributedSampler` helpers.
- **`src/train.py`**: Distributed training runner that imports `models` and `dataset`, configures the `torch_tpu` backend, wraps the model in PyTorch DDP (`DistributedDataParallel`), and executes the multi-epoch training loop.

Below, we create these modular files in a local `src/` directory.

In [ ]:
import pathlib

src_dir = pathlib.Path("src")
src_dir.mkdir(exist_ok=True)

# 1. Model architecture module (src/models.py)
models_code = '''"""Model architecture definitions."""
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLPClassifier(nn.Module):
    """Multi-layer perceptron classifier for image classification."""
    def __init__(self, in_features: int = 784, hidden1: int = 256, hidden2: int = 128, num_classes: int = 10):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)
'''

# 2. Dataset and data loader module (src/dataset.py)
dataset_code = '''"""Synthetic dataset generation and distributed DataLoader helpers."""
from typing import Tuple
import numpy as np
import torch
from torch.utils.data import DataLoader, DistributedSampler, TensorDataset

def generate_synthetic_data(
    num_samples: int = 16384,
    num_features: int = 784,
    num_classes: int = 10,
    seed: int = 42,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Generates synthetic clustered data for classification benchmarks."""
    np.random.seed(seed)
    prototypes = np.random.randn(num_classes, num_features).astype(np.float32)
    labels = np.random.randint(0, num_classes, size=(num_samples,))
    noise = np.random.randn(num_samples, num_features).astype(np.float32) * 0.5
    images = prototypes[labels] + noise
    return torch.tensor(images, dtype=torch.float32), torch.tensor(labels, dtype=torch.long)

def get_distributed_dataloader(
    batch_size: int = 128,
    num_samples: int = 16384,
    world_size: int = 1,
    rank: int = 0,
) -> Tuple[DataLoader, DistributedSampler]:
    """Builds a sharded DataLoader using DistributedSampler for DDP training."""
    images, labels = generate_synthetic_data(num_samples=num_samples, seed=42)
    dataset = TensorDataset(images, labels)
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)
    loader = DataLoader(dataset, batch_size=batch_size, sampler=sampler)
    return loader, sampler
'''

# 3. Main distributed training entrypoint (src/train.py)
train_code = '''#!/usr/bin/env python3
"""Multi-host distributed PyTorch DDP training entrypoint for TorchTPU."""
import functools
import logging
import os
import time
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.optim as optim
from torch.nn.parallel import DistributedDataParallel as DDP

# Import modular libraries mounted via Kubernetes ConfigMap
from dataset import get_distributed_dataloader
from models import MLPClassifier

# Ensure torch_tpu backend is registered
try:
    import torch_tpu
except ImportError:
    pass

logger = logging.getLogger(__name__)


def _maybe_init_distributed_on_gke(
    slicebuilder_first_worker_port: int = 10000,
    environ: dict[str, str] | None = None,
) -> bool:
    """Initializes environment variables for distributed TPU training on GKE.

    This function configures environment variables required by SliceBuilder
    when running torch_tpu on GKE, based on the provided environment
    variables set by the Kubernetes setup. It adjusts bounds and addresses
    to match the expected configuration for distributed TPU workers.

    Args:
        slicebuilder_first_worker_port: The starting port number for SliceBuilder
            worker communication.
        environ: A dictionary of environment variables. Defaults to `os.environ`.

    Returns:
        True if the environment variables were successfully initialized for
        distributed TPU training on GKE, False otherwise.
    """
    if environ is None:
        environ = os.environ

    if (
        "TPU_WORKER_HOSTNAMES" not in environ
        or "TPU_CHIPS_PER_HOST_BOUNDS" not in environ
        or "TPU_HOST_BOUNDS" not in environ
        or "WORLD_SIZE" not in environ
        or "LOCAL_RANK" not in environ
        or "RANK" not in environ
    ):
        return False

    logger.info("Trying to initialize distributed TPU training on GKE.")
    world_size = int(environ["WORLD_SIZE"])
    local_rank = int(environ["LOCAL_RANK"])
    rank = int(environ["RANK"])
    is_v7 = (
        "TPU_ACCELERATOR_TYPE" in environ
        and environ["TPU_ACCELERATOR_TYPE"].startswith("tpu7x")
    )

    tpu_chips_per_host_bounds = list(
        map(int, environ["TPU_CHIPS_PER_HOST_BOUNDS"].split(","))
    )
    tpu_chips_per_host_bounds_product = functools.reduce(
        lambda x, y: x * y, tpu_chips_per_host_bounds
    )

    if tpu_chips_per_host_bounds_product == 1:
        return False  # either single host or environment is set manually.

    tpu_host_bounds = list(map(int, environ["TPU_HOST_BOUNDS"].split(",")))
    tpu_host_bounds_product = functools.reduce(
        lambda x, y: x * y, tpu_host_bounds
    )

    tpu_worker_hostnames = environ["TPU_WORKER_HOSTNAMES"].split(",")

    if tpu_host_bounds_product != len(tpu_worker_hostnames):
        return False  # environment is likely to set incorrectly.

    if is_v7:
        tpu_chips_per_host_bounds_product *= 2

    if tpu_chips_per_host_bounds_product * tpu_host_bounds_product != world_size:
        return False  # environment is likely to set incorrectly.

    environ["TPU_CHIPS_PER_HOST_BOUNDS"] = ",".join(
        ["1"] * (4 if is_v7 else 3)
    )
    for i in range(len(tpu_host_bounds)):
        tpu_host_bounds[i] = tpu_host_bounds[i] * tpu_chips_per_host_bounds[i]
    environ["TPU_HOST_BOUNDS"] = ",".join(map(str, tpu_host_bounds))
    if is_v7:
        environ["TPU_HOST_BOUNDS"] += ",2"

    environ["TPU_VISIBLE_CHIPS"] = str(local_rank)
    environ["CLOUD_TPU_TASK_ID"] = str(rank)
    ports = list(
        range(
            slicebuilder_first_worker_port,
            slicebuilder_first_worker_port + tpu_chips_per_host_bounds_product,
        )
    )
    environ["TPU_PROCESS_PORT"] = str(ports[local_rank])
    tpu_worker_addresses = []
    for tpu_worker_hostname in tpu_worker_hostnames:
        for port in ports:
            tpu_worker_addresses.append(tpu_worker_hostname + ":" + str(port))
    tpu_worker_addresses_str = ",".join(tpu_worker_addresses)
    environ["TPU_PROCESS_ADDRESSES"] = tpu_worker_addresses_str
    environ["TORCH_TPU_SLICEBUILDER_ADDRESSES"] = tpu_worker_addresses_str
    environ["TORCH_TPU_TOPOLOGY"] = environ["TPU_HOST_BOUNDS"]
    return True


def main():
    # 1. Initialize GKE multi-host TPU environment and process group
    _maybe_init_distributed_on_gke()
    if not dist.is_initialized():
        dist.init_process_group(backend="tpu_dist")
    device = torch.device("tpu")

    rank = dist.get_rank()
    world_size = dist.get_world_size()
    host_id = os.environ.get("TPU_WORKER_ID", os.environ.get("NODE_RANK", "0"))
    local_rank = os.environ.get("LOCAL_RANK", "0")

    print(
        f"[Host {host_id} | LocalRank {local_rank} | GlobalRank {rank}/{world_size}] "
        f"Initialized TorchTPU DDP worker on device: {device} "
        f"(TORCH_TPU_TOPOLOGY={os.environ.get(\'TORCH_TPU_TOPOLOGY\')})",
        flush=True,
    )

    # 2. Synchronize random seeds for model weights across all ranks
    torch.manual_seed(42)

    # 3. Model setup on TPU wrapped in DDP
    model = MLPClassifier().to(device=device, dtype=torch.bfloat16)
    ddp_model = DDP(model)

    # 4. Distributed DataLoader sharded across all ranks
    loader, sampler = get_distributed_dataloader(
        batch_size=128,
        num_samples=16384,
        world_size=world_size,
        rank=rank,
    )

    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.SGD(ddp_model.parameters(), lr=0.05)

    epochs = 5
    total_start = time.time()

    for epoch in range(1, epochs + 1):
        sampler.set_epoch(epoch)
        epoch_start = time.time()
        running_loss = 0.0
        total_samples = 0

        for batch_images, batch_labels in loader:
            batch_images = batch_images.to(device=device, dtype=torch.bfloat16)
            batch_labels = batch_labels.to(device=device)

            optimizer.zero_grad()
            outputs = ddp_model(batch_images)
            loss = loss_fn(outputs, batch_labels)
            loss.backward()
            optimizer.step()

            # Execute .item() synchronously across all ranks to prevent divergence
            current_loss = loss.detach().item()
            running_loss += current_loss * batch_images.size(0)
            total_samples += batch_images.size(0)

        epoch_loss = running_loss / total_samples
        duration = time.time() - epoch_start
        throughput = (total_samples * world_size) / duration

        if rank == 0:
            print(
                f"Epoch {epoch:2d}/{epochs:2d} | Avg Loss: {epoch_loss:.4f} | "
                f"Throughput: {throughput:.0f} samples/s ({duration:.2f}s)",
                flush=True,
            )

    if rank == 0:
        print(
            f"\\n[Rank 0] Multi-host distributed training successfully finished "
            f"in {time.time() - total_start:.2f}s across {world_size} TPU cores!",
            flush=True,
        )

    dist.destroy_process_group()

if __name__ == "__main__":
    main()
'''

(src_dir / "models.py").write_text(models_code)
(src_dir / "dataset.py").write_text(dataset_code)
(src_dir / "train.py").write_text(train_code)

print("Modular training code written to src/:")
for f in sorted(src_dir.glob("*.py")):
    print(f" - {f.name} ({f.stat().st_size} bytes)")

## 3. Submit Multi-Host Distributed Training (`submit_multihost_training`)

All Kubernetes plumbing (ConfigMap packaging, volume mounts, `RuntimePatch`, nodeSelectors, tolerations, and `torchrun` command configuration) is encapsulated inside `tpu_trainer.py`. First, write `tpu_trainer.py` to the workspace using `%%writefile`, then call `submit_multihost_training()` with high-level training parameters:
- **`image`**: Container image containing PyTorch and TorchTPU.
- **`src_dir`**: Directory containing your modular library files (`"src"`).
- **`main_script`**: Training entrypoint script (`"train.py"`).
- **`num_nodes`**: Number of TPU host nodes in the slice (`2`).
- **`tpus_per_node`**: Physical TPU chips per host (`4`).
- **`compute_class`**: GKE TPU ComputeClass (`"tpu-v5-8-multi-host"`).

*(Note: Inside each worker process, `_maybe_init_distributed_on_gke()` automatically derives `TORCH_TPU_SLICEBUILDER_ADDRESSES`, `TORCH_TPU_TOPOLOGY`, `TPU_HOST_BOUNDS`, `TPU_CHIPS_PER_HOST_BOUNDS`, `TPU_VISIBLE_CHIPS`, `CLOUD_TPU_TASK_ID`, `TPU_PROCESS_PORT`, and `TPU_PROCESS_ADDRESSES` from GKE's injected `TPU_WORKER_HOSTNAMES`, `TPU_HOST_BOUNDS`, `TPU_CHIPS_PER_HOST_BOUNDS`, and `torchrun`'s rank environment variables.)*

In [ ]:
%%writefile tpu_trainer.py
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""High-level abstraction for launching multi-host TorchTPU training jobs on GKE.

Encapsulates all Kubernetes infrastructure details (ConfigMaps, volume mounts,
node selectors, tolerations, and Kubeflow Trainer SDK patches) so data
scientists can launch distributed training jobs with a single Python function.
"""

import os
import pathlib
import time
from typing import Dict, Optional, Union

from kubernetes import client as k8s_client, config as k8s_config
from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import CustomTrainerContainer, TrainerClient
from kubeflow.trainer.options import (
    ContainerPatch,
    JobSetSpecPatch,
    JobSetTemplatePatch,
    JobSpecPatch,
    JobTemplatePatch,
    Name,
    PodSpecPatch,
    PodTemplatePatch,
    ReplicatedJobPatch,
    RuntimePatch,
    TrainerCommand,
    TrainingRuntimeSpecPatch,
)


def get_current_namespace() -> str:
    """Discovers the active Kubernetes namespace from environment or serviceaccount."""
    if os.environ.get("TENANT_NAMESPACE"):
        return os.environ["TENANT_NAMESPACE"]
    ns_path = "/var/run/secrets/kubernetes.io/serviceaccount/namespace"
    if os.path.exists(ns_path):
        with open(ns_path, encoding="utf-8") as f:
            return f.read().strip()
    return "default"


def _init_k8s_clients():
    """Initializes in-cluster or kube-config clients."""
    try:
        k8s_config.load_incluster_config()
    except Exception:
        k8s_config.load_kube_config()
    return k8s_client.CoreV1Api(), k8s_client.CustomObjectsApi()


def submit_multihost_training(
    image: str,
    src_dir: Union[str, pathlib.Path] = "src",
    main_script: str = "train.py",
    num_nodes: int = 2,
    tpus_per_node: int = 4,
    compute_class: str = "tpu-v5-8-multi-host",
    namespace: Optional[str] = None,
    job_name: Optional[str] = None,
    extra_env: Optional[Dict[str, str]] = None,
) -> str:
    """Packages local training code and submits a multi-host TrainJob on GKE.

    Args:
        image: Container image containing PyTorch and TorchTPU.
        src_dir: Directory containing user training scripts and modules.
        main_script: Entrypoint script relative to src_dir (e.g. "train.py").
        num_nodes: Number of TPU host nodes in the slice (e.g. 2).
        tpus_per_node: Physical TPU chips per host node (e.g. 4).
        compute_class: GKE TPU ComputeClass (e.g. "tpu-v5-8-multi-host").
        namespace: Target Kubernetes namespace.
        job_name: Optional custom job name (defaults to timestamped name).
        extra_env: Optional dictionary of additional environment variables.

    Returns:
        The submitted TrainJob name.
    """
    if not image:
        raise ValueError("An 'image' must be provided for the TPU training container.")
    namespace = namespace or get_current_namespace()
    job_name = job_name or f"torch-tpu-multihost-{int(time.time())}"
    core_v1, _ = _init_k8s_clients()

    # 1. Package source code directory into a Kubernetes ConfigMap
    src_path = pathlib.Path(src_dir)
    if not src_path.exists():
        raise FileNotFoundError(f"Source directory '{src_dir}' not found.")

    cm_name = f"{job_name}-code"
    cm_data = {
        file_path.name: file_path.read_text(encoding="utf-8")
        for file_path in sorted(src_path.glob("*.py"))
    }
    if not cm_data:
        raise ValueError(f"No Python files found in '{src_dir}'.")
    if main_script not in cm_data:
        raise ValueError(
            f"Main script '{main_script}' not found in '{src_dir}' ({list(cm_data.keys())})."
        )

    cm_body = k8s_client.V1ConfigMap(
        metadata=k8s_client.V1ObjectMeta(name=cm_name, namespace=namespace),
        data=cm_data,
    )
    core_v1.create_namespaced_config_map(namespace=namespace, body=cm_body)
    print(f"[K8s] Created ConfigMap '{cm_name}' with {len(cm_data)} files: {list(cm_data.keys())}")

    # 2. Define Trainer container specification
    trainer = CustomTrainerContainer(
        image=image,
        num_nodes=num_nodes,
        resources_per_node={"google.com/tpu": str(tpus_per_node)},
        env=extra_env,
    )

    # 3. Attach GKE accelerator patch and ConfigMap volume mount
    tpu_patch = RuntimePatch(
        training_runtime_spec=TrainingRuntimeSpecPatch(
            template=JobSetTemplatePatch(
                spec=JobSetSpecPatch(
                    replicated_jobs=[
                        ReplicatedJobPatch(
                            name="node",
                            template=JobTemplatePatch(
                                spec=JobSpecPatch(
                                    template=PodTemplatePatch(
                                        spec=PodSpecPatch(
                                            node_selector={
                                                "cloud.google.com/compute-class": compute_class,
                                            },
                                            tolerations=[
                                                {
                                                    "key": "google.com/tpu",
                                                    "operator": "Exists",
                                                    "effect": "NoSchedule",
                                                },
                                                {
                                                    "key": "cloud.google.com/compute-class",
                                                    "operator": "Exists",
                                                    "effect": "NoSchedule",
                                                },
                                            ],
                                            volumes=[
                                                {
                                                    "name": "code-volume",
                                                    "configMap": {
                                                        "name": cm_name,
                                                    },
                                                }
                                            ],
                                            containers=[
                                                ContainerPatch(
                                                    name="node",
                                                    volume_mounts=[
                                                        {
                                                            "name": "code-volume",
                                                            "mountPath": "/workspace",
                                                        }
                                                    ],
                                                )
                                            ],
                                        )
                                    )
                                )
                            )
                        )
                    ]
                )
            )
        )
    )

    # 4. Submit via Kubeflow TrainerClient
    # Note: Kubeflow Trainer's 'torch-distributed' runtime automatically injects
    # PET_NNODES, PET_NODE_RANK, PET_MASTER_ADDR, and PET_MASTER_PORT, which
    # torchrun reads natively.
    trainer_client = TrainerClient(
        backend_config=KubernetesBackendConfig(namespace=namespace)
    )
    submitted_name = trainer_client.train(
        runtime="torch-distributed",
        trainer=trainer,
        options=[
            Name(job_name),
            TrainerCommand(
                [
                    "torchrun",
                    f"--nproc_per_node={tpus_per_node}",
                    f"/workspace/{main_script}",
                ]
            ),
            tpu_patch,
        ],
    )
    print(f"Successfully submitted multi-host TrainJob '{submitted_name}' (hosts={num_nodes}, chips={num_nodes * tpus_per_node})")
    return submitted_name


def wait_for_job_pods(
    job_name: str,
    num_nodes: int = 2,
    timeout: int = 600,
    namespace: Optional[str] = None,
) -> bool:
    """Monitors worker Pods until all reach Running or Succeeded status."""
    namespace = namespace or get_current_namespace()
    core_v1, _ = _init_k8s_clients()

    print(f"Waiting for worker pods of TrainJob '{job_name}' to become Running...")
    start_time = time.time()

    while time.time() - start_time < timeout:
        pods = core_v1.list_namespaced_pod(
            namespace=namespace,
            label_selector=f"jobset.sigs.k8s.io/jobset-name={job_name}",
        ).items
        if len(pods) >= num_nodes:
            status_str = ", ".join(
                f"{p.metadata.name.split('-')[-3]}-{p.metadata.name.split('-')[-2]}: {p.status.phase}"
                for p in pods
            )
            print(f"[{int(time.time() - start_time):3d}s] Pod statuses: {status_str}", flush=True)
            if all(p.status.phase in ("Running", "Succeeded") for p in pods):
                print(f"\nAll {num_nodes} worker pods are active and Running!", flush=True)
                return True
        else:
            print(f"[{int(time.time() - start_time):3d}s] Waiting for pods to be created...", flush=True)
        time.sleep(5)

    print("\nWarning: Not all pods reached Running phase within timeout.")
    return False


def stream_job_logs(
    job_name: str,
    num_nodes: int = 2,
    max_iterations: int = 60,
    namespace: Optional[str] = None,
):
    """Streams and filters distributed logs from all worker pods."""
    namespace = namespace or get_current_namespace()
    core_v1, _ = _init_k8s_clients()

    print(f"Streaming distributed training logs for TrainJob '{job_name}'...\n")
    completed_phases = {"Succeeded", "Failed"}
    logged_lines = {}

    for _ in range(max_iterations):
        pods = core_v1.list_namespaced_pod(
            namespace=namespace,
            label_selector=f"jobset.sigs.k8s.io/jobset-name={job_name}",
        ).items

        all_completed = len(pods) >= num_nodes and all(p.status.phase in completed_phases for p in pods)

        for p in sorted(pods, key=lambda x: x.metadata.name):
            pod_name = p.metadata.name
            if pod_name not in logged_lines:
                logged_lines[pod_name] = 0
            try:
                raw_log = core_v1.read_namespaced_pod_log(
                    name=pod_name,
                    namespace=namespace,
                    container="node",
                )
                lines = raw_log.splitlines()
                if len(lines) > logged_lines[pod_name]:
                    new_lines = lines[logged_lines[pod_name]:]
                    logged_lines[pod_name] = len(lines)
                    for line in new_lines:
                        if any(skip in line for skip in ["cachednslookup", "compilation_cache", "OMP_NUM_THREADS", "TcMallocWarning"]):
                            continue
                        prefix = f"[{pod_name.split('-')[-3]}-{pod_name.split('-')[-2]}]"
                        print(f"{prefix:12s} {line}", flush=True)
            except Exception:
                pass

        if all_completed:
            print(f"\nAll {num_nodes} worker pods have successfully finished execution!", flush=True)
            break

        time.sleep(3)


def delete_training_job(
    job_name: str,
    namespace: Optional[str] = None,
    delete_configmap: bool = True,
):
    """Deletes the TrainJob and associated code ConfigMap."""
    namespace = namespace or get_current_namespace()
    core_v1, _ = _init_k8s_clients()
    trainer_client = TrainerClient(
        backend_config=KubernetesBackendConfig(namespace=namespace)
    )

    try:
        trainer_client.delete_job(job_name)
        print(f"Deleted TrainJob: {job_name}")
    except Exception as e:
        print(f"Error deleting TrainJob: {e}")

    if delete_configmap:
        cm_name = f"{job_name}-code"
        try:
            core_v1.delete_namespaced_config_map(name=cm_name, namespace=namespace)
            print(f"Deleted ConfigMap: {cm_name}")
        except Exception:
            pass

In [ ]:
from tpu_trainer import submit_multihost_training

# TPU training container image with PyTorch and torch_tpu
image = os.environ.get("TORCH_TPU_IMAGE") or f"{os.environ.get('REGISTRY', '').rstrip('/')}/jupyterlab:latest-tpu"

# Submit the distributed training job with a single Python call
job_name = submit_multihost_training(
    image=image,
    src_dir="src",
    main_script="train.py",
    num_nodes=2,
    tpus_per_node=4,
    compute_class="tpu-v5-8-multi-host",
)

## 4. Monitor TrainJob & Active Host Pods

Watch the `TrainJob` and inspect the worker Pods scheduled across the TPU nodes. Once all host pods transition to `Running`, distributed training begins.

In [ ]:
from tpu_trainer import wait_for_job_pods

# Wait until all host pods are active and Running
wait_for_job_pods(job_name, num_nodes=2)

## 5. Stream Distributed Logs Across TPU Hosts

Stream output from all active worker pods. Observe how each host initializes its local TorchTPU environment, exchanges SliceBuilder addresses, launches its 4 local TPU processes via `torchrun`, and executes distributed training across all 8 cores.

In [ ]:
from tpu_trainer import stream_job_logs

# Stream and filter distributed training logs across all TPU worker nodes
stream_job_logs(job_name, num_nodes=2)

## 6. Cleanup Resources

When you have finished inspecting the distributed training logs, execute this cell to delete the `TrainJob` and the associated code `ConfigMap`.

In [ ]:
from tpu_trainer import delete_training_job

# Clean up TrainJob and the uploaded code ConfigMap
delete_training_job(job_name)